# CozyDay P1 A8 — Stage 1
**Owner: Lin Gao** | Step 1.1 Knowledge Base and Step 1.2 Chunking only. Stages 2–4 will be added by teammates after Stage 1 is merged.

## 1. Load and validate the knowledge base
30 original English paragraphs with stable IDs, titles, topics, and bodies. All filesystem paths are relative to this notebook / module directory.

In [4]:
from chunking import (load_knowledge_base, validate_knowledge_base,
                      fixed_length_chunks, overlapping_paragraph_chunks, hybrid_topic_chunks,
                      build_all_chunks, validate_chunks)
records = load_knowledge_base()
validation = validate_knowledge_base(records)
print(validation)
print("First paragraph:", records[0]["id"], records[0]["title"])

{'paragraphs': 30, 'total_words': 3111, 'min_words': 97, 'max_words': 112, 'unique_ids': 30}
First paragraph: P001 Creating an actionable task


## 2. Three deterministic chunking strategies
**Fixed:** greedy target of 130 words without splitting sentences. **Overlap:** consecutive full paragraph pairs with stride 1. **Hybrid:** packs only adjacent paragraphs sharing the same topic, soft cap 230 words.

In [5]:
chunks_by_strategy = build_all_chunks(records)
for strategy, chunks in chunks_by_strategy.items():
    print(f"{strategy}: {len(chunks)} chunks; first source IDs: {chunks[0]['source_ids']}")
    print("Example:", chunks[0]["text"][:350], "...\n")

fixed: 26 chunks; first source IDs: ['P001']
Example: A useful task describes one clear outcome rather than a vague intention. For example, replace “work on biology” with “summarize chapter four and answer the practice questions.” When preparing a task in CozyDay, give it a short descriptive title, optional notes, an estimated duration, and a realistic due date if one exists. Record the next physical  ...

overlap: 29 chunks; first source IDs: ['P001', 'P002']
Example: A useful task describes one clear outcome rather than a vague intention. For example, replace “work on biology” with “summarize chapter four and answer the practice questions.” When preparing a task in CozyDay, give it a short descriptive title, optional notes, an estimated duration, and a realistic due date if one exists. Record the next physical  ...

hybrid: 15 chunks; first source IDs: ['P001', 'P002']
Example: A useful task describes one clear outcome rather than a vague intention. For example, replace “work on biol

## 3. Automated checks and interface for Stage 2
Every strategy must preserve complete source content and produce nonempty chunks. Stage 2 should use `chunk["text"]` for embeddings and retain `chunk_id`/`source_ids`.

In [6]:
for strategy, chunks in chunks_by_strategy.items():
    assert validate_chunks(records, chunks, strategy)
    assert chunks and all(chunk["text"].strip() for chunk in chunks)
print("Stage 1 checks PASSED")
print("Chunk object keys:", list(chunks_by_strategy["fixed"][0]))

Stage 1 checks PASSED
Chunk object keys: ['chunk_id', 'strategy', 'source_ids', 'topic', 'text', 'word_count']
